In [2]:
import pandas as pd
import re
import nltk
import spacy
import seaborn as sns
import matplotlib.pyplot as plt
from nltk import word_tokenize
import wordcloud

In [31]:
df = pd.read_json("News_Category_Dataset_v3.json",lines=True)
#df.head()

In [ ]:
df.info()

keep headline, category, short description, take 4000> category

Merge categories

In [32]:
df['category'] = df['category'].replace({"HEALTHY LIVING": "WELLNESS",
"QUEER VOICES": "GROUPS VOICES",
"BUSINESS": "BUSINESS & FINANCES",
"PARENTS": "PARENTING",
"BLACK VOICES": "GROUPS VOICES",
"THE WORLDPOST": "WORLD NEWS",
"STYLE": "STYLE & BEAUTY",
"GREEN": "ENVIRONMENT",
"TASTE": "LIFESTYLE",
"WORLDPOST": "WORLD NEWS",
"SCIENCE": "SCIENCE & TECH",
"TECH": "SCIENCE & TECH",
"MONEY": "BUSINESS & FINANCES",
"ARTS": "ARTS & CULTURE",
"COLLEGE": "EDUCATION",
"LATINO VOICES": "GROUPS VOICES",
"CULTURE & ARTS": "ARTS & CULTURE",
"FIFTY": "MISCELLANEOUS",
"GOOD NEWS": "MISCELLANEOUS",
"TRAVEL":"LIFESTYLE"})

"""df['category'] = df['category'].replace({
    'WELLNESS':'LIFESTYLE',
    'STYLE & BEAUTY':'LIFESTYLE',
    'GROUPS VOICES':'OPINION/EDITORIAL',
    'WORLD NEWS': 'WORLD',
    'BUSINESS & FINANCES': 'BUSINESS',
    'COMEDY':'ENTERTAINMENT',
    'SPORTS':'ENTERTAINMENT',
    'HOME & LIVING':'LIFESTYLE',
    'PARENTING':'RELATIONSHIPS',
    'SCIENCE & TECH':'SCIENCE',
    'WEDDINGS':'RELATIONSHIPS',
    'ENVIRONMENT':'SCIENCE',
    'DIVORCE':'RELATIONSHIPS',
    'ARTS & CULTURE':'ENTERTAINMENT',
    'WOMEN':'OPINION/EDITORIAL',
    'IMPACT':'OPINION/EDITORIAL',
    'CRIME':'LOCAL',
    'MEDIA':'ENTERTAINMENT',
    'WEIRD NEWS':'MISCELLANOUS',
    'RELIGION':'OPINION/EDITORIAL',
    'U.S. NEWS':'LOCAL',
    'MISCELLANEOUS':'MISCELLANOUS'

})"""

In [ ]:
df['category'].value_counts()

In [ ]:
plt.figure(figsize=(25,13))
sns.barplot(y=df['category'].value_counts().values,x=df['category'].value_counts().index)
plt.title("The distribution of categories")
plt.xlabel("Categories of articles")
plt.ylabel("The number of articles")

plt.yticks(rotation=0,fontsize = 16)
plt.show()

In [39]:
count_df = pd.DataFrame(df.groupby(["category"]).count()["headline"].sort_values(ascending=False),columns=["headline"])

In [ ]:
#get the category with number of news more than 4000
category_list = count_df.loc[count_df["headline"] >= 7000].index.to_list()
category_list

In [ ]:
#create a df with 7000 instances of each

In [56]:
sample_data = []
sample_df = pd.DataFrame()
for x in category_list:
    category_df = df[df['category']==x]

    if len(category_df)>=7000:
        sample_ins = category_df.sample(n=7000,random_state=42)
    else:
        sample_ins = category_df
    sample_data.append(sample_ins)
sample_df = pd.concat(sample_data,ignore_index = True)


In [53]:
#sample_df.drop(columns=['link','authors','date'],inplace=True)
sample_df.drop(columns=['full_news'],inplace=True)

In [57]:
#cleaning headline and short description
sample_df.drop_duplicates(['headline','short_description'],keep='first',inplace=True)
sample_df[sample_df['headline'] == ''].value_counts().sum()
sample_df = sample_df[~(sample_df['headline']=='')]
sample_df[sample_df['short_description'] == ''].value_counts().sum()
sample_df = sample_df[~(sample_df['short_description']=='')]
sample_df['full_news']  = sample_df['headline'] + sample_df['short_description']
sample_df['full_news'].isnull().sum()


0

2. entity information: most words
1. text preprocessing( Function to do text preprocessing: remove number, remove punctuation, remove stop_words, convert to lower case, and make lemmatization)
3. 

TEXT preprocessing (full_news)

In [58]:
sample_df['full_news'] = sample_df['full_news'].str.lower()
sample_df["full_news"] = sample_df["full_news"].apply(lambda x: re.sub(r'[^a-zA-Z\s]', " ", x))
sample_df['full_news'].tail()

62995    money laundering banks should be put out of bu...
62996    listen up  these    changes to financial rules...
62997    warren buffett  berkshire hathaway board  soli...
62998       charts that will restore your faith in the ...
62999    what the big airplane decision reveals about y...
Name: full_news, dtype: object

In [60]:
sample_df['full_news'] = sample_df['full_news'].apply(lambda x: nltk.word_tokenize(x))
sample_df['full_news'].tail()

62995    [money, laundering, banks, should, be, put, ou...
62996    [listen, up, these, changes, to, financial, ru...
62997    [warren, buffett, berkshire, hathaway, board, ...
62998    [charts, that, will, restore, your, faith, in,...
62999    [what, the, big, airplane, decision, reveals, ...
Name: full_news, dtype: object

In [61]:
#stopwords
stopwords = nltk.corpus.stopwords.words('english')
sample_df['full_news'] = sample_df['full_news'].apply(lambda x: [w for w in x if w.lower() not in stopwords] )


In [ ]:
#pip install spacy

In [82]:
sample_df["full_news"].head()

0    [u, lawmakers, join, demand, puerto, rico, gov...
4    [donald, trump, loved, cite, nonpartisan, agen...
5    [transit, cop, investigated, quizzing, passeng...
6    [watch, democratic, national, convention, live...
7    [judge, says, injunction, clean, water, rule, ...
Name: full_news, dtype: object

In [90]:
#Information extraction and wordcloud

#spacy.cli.download("en_core_web_lg")
#ner = spacy.load("en_core_web_lg")
#sample_df["tags"] = sample_df['full_news'].apply(lambda x: [(tag.text, tag.label_) for text in x for tag in ner(text).ents])

# Display the "tags" column
print(sample_df['tags'].head())

0      [(puerto, GPE), (ricans, ORG), (rossello, GPE)]
4    [(donald, PERSON), (trump, PERSON), (republica...
5                                 [(minneapolis, GPE)]
6    [(democratic, NORP), (livesen, PERSON), (eliza...
7                        [(epa, ORG), (erickson, ORG)]
Name: tags, dtype: object


In [95]:
sample_df.to_excel("C:/Users/Aditi/OneDrive/Desktop/Data Analysis/Internship/inshorts/ner.xlsx")

In [ ]:
#one hot encode by category
oh_enc.fit(data[['category']])
category_encoded = oh_enc.transform(data[['category']])


# If you want to convert the encoded data back to a DataFrame with appropriate column names
encoded_df = pd.DataFrame(category_encoded, columns=oh_enc.get_feature_names_out(['category']))

# Print the encoded DataFrame
print(encoded_df)

In [ ]:
Vectorize tokens
from sklearn.feature_extraction.text import TfidfVectorizer
   
vectorizer = TfidfVectorizer(max_features=3000, max_df=15, min_df=2)
data['token_desc'] = data['token_desc'].apply(lambda x: ' '.join(x))
# keep cell and output
vectorizer2 = TfidfVectorizer()
tfidf_matrix2 = vectorizer2.fit_transform(data['token_desc'])

# Convert the TF-IDF matrix to a DataFrame for easier viewing
tfidf_df2 = pd.DataFrame(tfidf_matrix2.toarray(), columns=vectorizer2.get_feature_names_out())

print(tfidf_df2)

